In [3]:
import numpy as np
import cv2
pixel_coordinates = [[202,63], [323,76], [256,112], [388,108], [232,161], [323,178], [308,190], [259,240], [388,239], [243,276], [323,274]]
robot_coordinates = [[393.6,-47.6], [245.8,-29.9], [325.3,11.4], [164.9,7.9], [356.9,73.4], [244.5,92.9], [262.6,107.8], [323.9,170.4], [164.6,168.5], [343.1,215.1], [243.9,212.2]]
pixel_points = np.array(pixel_coordinates, dtype=np.float32)
robot_points = np.array(robot_coordinates, dtype=np.float32)

homography_matrix, _ = cv2.findHomography(pixel_points, robot_points)
print("Homography Matrix:", homography_matrix)

Homography Matrix: [[-1.22081701e+00 -2.28299907e-02  6.44646305e+02]
 [ 5.36334473e-03  1.22986813e+00 -1.26542438e+02]
 [ 6.69144025e-05 -7.73294942e-05  1.00000000e+00]]


In [4]:
def pixel_to_robot(x, y, matrix):
    pixel = np.array([x, y, 1]).reshape(3, 1)
    robot_coords = np.dot(matrix, pixel)
    robot_coords /= robot_coords[2]  
    return robot_coords[0][0], robot_coords[1][0]

In [5]:
box = pixel_to_robot(306, 303, homography_matrix)
box_x, box_y = box

In [6]:
import cv2
import numpy as np
from xarm.wrapper import XArmAPI


arm = XArmAPI('192.168.1.155')
arm.motion_enable(enable=True)
arm.set_mode(0)
arm.set_state(0)
arm.connect()
arm.move_gohome()

SDK_VERSION: 1.14.8
ROBOT_IP: 192.168.1.155, VERSION: v2.2.0, PROTOCOL: V1, DETAIL: 6,9,LI1006,DL1000,v2.2.0, TYPE1300: [0, 0]
change protocol identifier to 3


0

ControllerError, code: 2
ControllerError had clean
ControllerError, code: 2
servo_error_code, servo_id=1, status=3, code=0
servo_error_code, servo_id=2, status=3, code=0
servo_error_code, servo_id=3, status=3, code=0
servo_error_code, servo_id=4, status=3, code=0
servo_error_code, servo_id=5, status=3, code=0
servo_error_code, servo_id=6, status=3, code=0
ControllerError had clean
servo_error_code, servo_id=1, status=0, code=0
servo_error_code, servo_id=2, status=0, code=0
servo_error_code, servo_id=3, status=0, code=0
servo_error_code, servo_id=4, status=0, code=0
servo_error_code, servo_id=5, status=0, code=0
servo_error_code, servo_id=6, status=0, code=0
ControllerError, code: 2
servo_error_code, servo_id=1, status=3, code=0
servo_error_code, servo_id=1, status=0, code=0
ControllerError had clean


In [7]:
import time
def pick_up_and_drop(x_robot, y_robot):
    arm.set_position(x_robot, y_robot, 70)  
    arm.set_position(x_robot, y_robot, 17, wait=True)  
    arm.set_suction_cup(False)
    time.sleep(0.5)
    arm.set_position(x_robot, y_robot, 16.5, wait=True)  
    arm.set_position(x_robot, y_robot, 70, wait=True)  
    arm.set_position(box_x, box_y, 200)
    arm.set_position(box_x, box_y, 100)
    arm.set_suction_cup(True)
    arm.set_position(box_x, box_y, 200)

In [8]:
import cv2

cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Can't receive frame. Exiting...")
        break

    cv2.imshow('Webcam Preview - Press Q to capture', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        cv2.imwrite('captured_image.jpg', frame)
        print("Image saved as captured_image.jpg")
        break

cap.release()
cv2.destroyAllWindows()

Image saved as captured_image.jpg


In [9]:
import dotenv
import os
from google import genai
from google.genai import types
from PIL import Image
import io
import os
import requests
from io import BytesIO

dotenv.load_dotenv()
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

model_name = "gemini-2.0-flash" 

bounding_box_system_instructions = """
    Return bounding box as a JSON array with labels. Never return masks or code fencing. Limit to only one object.
    If an object is present multiple times, name them according to their unique characteristic (colors, size, position, unique characteristics, etc..).
    """

client = genai.Client(api_key=GEMINI_API_KEY)

safety_settings = [
    types.SafetySetting(
        category="HARM_CATEGORY_DANGEROUS_CONTENT",
        threshold="BLOCK_ONLY_HIGH",
    ),
]

In [10]:
import json
import random
import io
from PIL import Image, ImageDraw, ImageFont
from PIL import ImageColor

additional_colors = [colorname for (colorname, colorcode) in ImageColor.colormap.items()]

def plot_bounding_boxes(im, bounding_boxes):
    """
    Plots bounding boxes on an image with markers for each a name, using PIL, normalized coordinates, and different colors.

    Args:
        img_path: The path to the image file.
        bounding_boxes: A list of bounding boxes containing the name of the object
         and their positions in normalized [y1 x1 y2 x2] format.
    """

    # Load the image
    img = im
    width, height = img.size
    print(img.size)
    # Create a drawing object
    draw = ImageDraw.Draw(img)

    # Define a list of colors
    colors = [
    'red',
    'green',
    'blue',
    'yellow',
    'orange',
    'pink',
    'purple',
    'brown',
    'gray',
    'beige',
    'turquoise',
    'cyan',
    'magenta',
    'lime',
    'navy',
    'maroon',
    'teal',
    'olive',
    'coral',
    'lavender',
    'violet',
    'gold',
    'silver',
    ] + additional_colors

    # Parsing out the markdown fencing
    bounding_boxes = parse_json(bounding_boxes)

    font = ImageFont.load_default()

    # Iterate over the bounding boxes
    for i, bounding_box in enumerate(json.loads(bounding_boxes)):
      # Select a color from the list
      color = colors[i % len(colors)]

      # Convert normalized coordinates to absolute coordinates
      abs_y1 = int(bounding_box["box_2d"][0]/1000 * height)
      abs_x1 = int(bounding_box["box_2d"][1]/1000 * width)
      abs_y2 = int(bounding_box["box_2d"][2]/1000 * height)
      abs_x2 = int(bounding_box["box_2d"][3]/1000 * width)

      if abs_x1 > abs_x2:
        abs_x1, abs_x2 = abs_x2, abs_x1

      if abs_y1 > abs_y2:
        abs_y1, abs_y2 = abs_y2, abs_y1

      # Draw the bounding box
      draw.rectangle(
          ((abs_x1, abs_y1), (abs_x2, abs_y2)), outline=color, width=4
      )

      # Draw the text
      if "label" in bounding_box:
        draw.text((abs_x1 + 8, abs_y1 + 6), bounding_box["label"], fill=color, font=font)

    # Display the image
    img.save("output_image.jpg")  # or "output_image.png"
    print("Image saved as output_image.jpg")

    img.show()

In [11]:
def parse_json(json_output):
    # Parsing out the markdown fencing
    lines = json_output.splitlines()
    for i, line in enumerate(lines):
        if line == "```json":
            json_output = "\n".join(lines[i+1:])  # Remove everything before "```json"
            json_output = json_output.split("```")[0]  # Remove everything after the closing "```"
            break  # Exit the loop once "```json" is found
    return json_output

In [12]:
im = None
def vision_transformer():
    global im
    prompt = "Detect the 2d bounding box for one small cube in the image outside the brown box. " \
    "choose the one with the highest probability"  # @param {type:"string"}

    image = "captured_image.jpg"
    # Load and resize image
    im = Image.open(io.BytesIO(open(image, "rb").read()))
    im.thumbnail([1024,1024], Image.Resampling.LANCZOS)

    # Run model to find bounding boxes
    response = client.models.generate_content(
        model=model_name,
        contents=[prompt, im],
        config = types.GenerateContentConfig(
            system_instruction=bounding_box_system_instructions,
            temperature=0.5,
        )
    )

    # plot_bounding_boxes(im, response.text)
    # Check output
    print(response.text)
    return response.text

In [13]:
import cv2
import json

# Initialize webcam
cap = cv2.VideoCapture(1)

while True:
    ret, frame = cap.read()

    if not ret:
        print("Can't receive frame. Exiting...")
        break

    # Save the current frame as an image (overwrite previous one)
    cv2.imwrite('captured_image.jpg', frame)

    # Process bounding boxes
    response = vision_transformer()
    bounding_boxes = parse_json(response)  # Parse JSON response
    bounding_boxes = json.loads(bounding_boxes)
    bounding_box = bounding_boxes[0]
    width, height = im.size
    # Convert normalized coordinates to absolute coordinates
    abs_y1 = int(bounding_box["box_2d"][0]/1000 * height)
    abs_x1 = int(bounding_box["box_2d"][1]/1000 * width)
    abs_y2 = int(bounding_box["box_2d"][2]/1000 * height)
    abs_x2 = int(bounding_box["box_2d"][3]/1000 * width)

    if abs_x1 > abs_x2:
        abs_x1, abs_x2 = abs_x2, abs_x1

    if abs_y1 > abs_y2:
        abs_y1, abs_y2 = abs_y2, abs_y1
    
    c1 = (abs_x1 + abs_x2) / 2
    c2 = (abs_y1 + abs_y2) / 2

    c1_robot, c2_robot = pixel_to_robot(c1, c2, homography_matrix)
    pick_up_and_drop(c1_robot, c2_robot)
    arm.move_gohome()

    # Display the frame
    cv2.imshow('Webcam Preview - Press Q to Exit', frame)

    # Check for 'q' key press to exit qloop
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
arm.move_gohome()

```json
[
  {"box_2d": [223, 504, 287, 554], "label": "cube"}
]
```
```json
[
  {"box_2d": [278, 353, 343, 398], "label": "small cube"}
]
```
```json
[
  {"box_2d": [610, 567, 662, 612], "label": "blue cube"}
]
```
```json
[
  {"box_2d": [739, 465, 797, 506], "label": "red cube"}
]
```
```json
[
  {"box_2d": [380, 453, 436, 498], "label": "small cube"}
]
```
```json
[
  {"box_2d": [256, 383, 304, 415], "label": "small cube"}
]
```
```json
[
  {"box_2d": [407, 394, 453, 436], "label": "red cube"}
]
```
```json
[
  {"box_2d": [668, 620, 883, 908], "label": "cube"}
]
```
[SDK][ERROR][2025-03-27 18:04:50][base.py:380] - - wait_feedback, xarm is stop, state=4
[SDK][ERROR][2025-03-27 18:04:50][base.py:380] - - API -> set_tgpio_digital(ionum=0, value=0) -> code=1
[SDK][ERROR][2025-03-27 18:04:50][base.py:380] - - API -> set_tgpio_digital(ionum=1, value=1) -> code=1
[SDK][ERROR][2025-03-27 18:04:50][base.py:380] - - API -> set_suction_cup(on=False, wait=False, delay_sec=None) -> code=1
[SDK][E

1